In [ ]:
# 01_eda.ipynb — Study 1 EDA
# Run in Colab. Loads study1_daily.csv via Kaggle API.

import pandas as pd
import matplotlib.pyplot as plt

# Credentials from Colab Secrets (key icon, left sidebar) -- never hardcode a real
# key here, this repo is public and it would stay in git history even if removed later.
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip
df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
print(df.shape)  # expect (2660, 144)
print(df["date"].min(), df["date"].max())

# --- Confirm target column ---
target_candidates = [
    "max_demand_met_total_mw",
    "evening_peak_demand_total_mw",
    "energy_met_total_mu",
]
for c in target_candidates:
    print(c, c in df.columns)

# --- Missingness ---
null_pct = df.isna().mean().sort_values(ascending=False) * 100
print(null_pct.head(20))

# --- Demand over time ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["max_demand_met_total_mw"], color="#2a78d6")
plt.title("National max demand met (MW) over time")
plt.show()

# --- Weekly/yearly seasonality ---
df["dow"] = df["date"].dt.dayofweek
df["year"] = df["date"].dt.year
df.groupby("dow")["max_demand_met_total_mw"].mean().plot(kind="bar", title="Avg demand by day of week", color="#eb6834")
plt.show()

df.groupby("year")["max_demand_met_total_mw"].mean().plot(kind="bar", title="Avg demand by year", color="#1baf7a")
plt.show()

# --- RES share trend (needed for Era 1 later) ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["share_res_pct"], color="#008300")
plt.title("RES share % over time")
plt.show()

# --- Known gaps: cross-check against the known-gaps list from Phase 0 ---
full_range = pd.date_range(df["date"].min(), df["date"].max())
missing_dates = full_range.difference(df["date"])
print(f"{len(missing_dates)} missing dates (expect ~69 known gaps -- see Pipeline/known_gaps.json)")
